<a href='http://www.oddbox.co.uk'> <img src='https://images.prismic.io/oddbox/Z1m1ppbqstJ98Vin_logo-1-.png?auto=format,compress' /></a>


## Stetch Goals Task

At Oddbox, customers can swap produce items in their box with one of N alternatives.
How would you approach incorporating this customisation logic into your demand forecast?

## Import Libraries

In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter, defaultdict
from pyvis.network import Network
import community as community_louvain

## Read and Clean Data

Data obtained from [Kaggle Blinkit Sales](https://www.kaggle.com/datasets/akxiit/blinkit-sales-dataset?select=blinkit_order_items.csv)
This data has MIT public sharing license

In [2]:
# load datasets
df_products = pd.read_csv('./Groceries_dataset/blinkit_products.csv')
df_orders = pd.read_csv('./Groceries_dataset/blinkit_orders.csv')
df_order_items = pd.read_csv('./Groceries_dataset/blinkit_order_items.csv')

In [3]:
# Clean strings
df_products['product_name'] = df_products['product_name'].str.strip()
df_products['brand'] = df_products['brand'].str.strip()
df_products['category'] = df_products['category'].str.strip()

# Create unique product key
df_products['product_key'] = df_products['product_name'] + " | " + df_products['brand']

# Product metadata
product_category = df_products[['product_key', 'category']].drop_duplicates().set_index('product_key')['category'].to_dict()
product_meta = df_products[['product_key', 'product_name', 'category', 'brand']].drop_duplicates().set_index('product_key')

## Analyse Data

In [4]:
df_products.head()

,product_id,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level,product_key
0,153019,Onions,Fruits & Vegetables,Aurora LLC,947.95,1263.93,25.0,3,13,88,Onions | Aurora LLC
1,11422,Potatoes,Fruits & Vegetables,Ramaswamy-Tata,127.16,169.55,25.0,3,20,65,Potatoes | Ramaswamy-Tata
2,669378,Potatoes,Fruits & Vegetables,Chadha and Sons,212.14,282.85,25.0,3,23,70,Potatoes | Chadha and Sons
3,848226,Tomatoes,Fruits & Vegetables,Barad and Sons,209.59,279.45,25.0,3,10,51,Tomatoes | Barad and Sons
4,890623,Onions,Fruits & Vegetables,"Sangha, Nagar and Varty",354.52,472.69,25.0,3,27,55,"Onions | Sangha, Nagar and Varty"


In [5]:
df_products[df_products['product_name']=='Milk']

,product_id,product_name,category,brand,price,mrp,margin_percentage,shelf_life_days,min_stock_level,max_stock_level,product_key
32,89084,Milk,Dairy & Breakfast,"Wali, Virk and Iyer",598.83,748.54,20.0,7,27,74,"Milk | Wali, Virk and Iyer"


In [6]:
df_orders.head()

,order_id,customer_id,order_date,promised_delivery_time,actual_delivery_time,delivery_status,order_total,payment_method,delivery_partner_id,store_id
0,1961864118,30065862,2024-07-17 08:34:01,2024-07-17 08:52:01,2024-07-17 08:47:01,On Time,3197.07,Cash,63230,4771
1,1549769649,9573071,2024-05-28 13:14:29,2024-05-28 13:25:29,2024-05-28 13:27:29,On Time,976.55,Cash,14983,7534
2,9185164487,45477575,2024-09-23 13:07:12,2024-09-23 13:25:12,2024-09-23 13:29:12,On Time,839.05,UPI,39859,9886
3,9644738826,88067569,2023-11-24 16:16:56,2023-11-24 16:34:56,2023-11-24 16:33:56,On Time,440.23,Card,61497,7917
4,5427684290,83298567,2023-11-20 05:00:39,2023-11-20 05:17:39,2023-11-20 05:18:39,On Time,2526.68,Cash,84315,2741


In [7]:
df_orders.shape

(5000, 10)

In [8]:
df_order_items.head()

,order_id,product_id,quantity,unit_price
0,1961864118,642612,3,517.03
1,1549769649,378676,1,881.42
2,9185164487,741341,2,923.84
3,9644738826,561860,1,874.78
4,5427684290,602241,2,976.55


In [9]:
df_order_items.shape

(5000, 4)

Create a single dataset with customer info, order info and product info

In [10]:
df_order_items_full_details = (df_order_items
                                .merge(df_orders[['order_id', 'customer_id', 'order_date']], how='left', on='order_id')
                                .merge(df_products[['product_id', 'product_name', 'category', 'brand', 'product_key']], how='left', on='product_id')
                                )
df_order_items_full_details['cus_transaction'] = df_order_items_full_details['customer_id'].astype(str) + '_' + pd.to_datetime(df_order_items_full_details['order_date']).dt.strftime('%Y-%m-%d')

We notice that each order gets a separate id even if it is by the same customer on the same day, so we use a concatenation of customer_id and date to derive a basket

In [11]:
df_order_items_full_details['cus_transaction'].value_counts(ascending=False)

cus_transaction
63402854_2024-05-30    2
75082508_2023-09-28    2
84119685_2024-10-30    2
119099_2024-05-24      2
75928073_2024-08-27    2
                      ..
43980719_2024-08-08    1
55042740_2024-03-15    1
51014003_2023-05-16    1
37904682_2023-06-07    1
28663279_2023-08-23    1
Name: count, Length: 4992, dtype: int64

In [12]:
df_order_items_full_details.head(15)

,order_id,product_id,quantity,unit_price,customer_id,order_date,product_name,category,brand,product_key,cus_transaction
0,1961864118,642612,3,517.03,30065862,2024-07-17 08:34:01,Pet Treats,Pet Care,Pillay-Ahuja,Pet Treats | Pillay-Ahuja,30065862_2024-07-17
1,1549769649,378676,1,881.42,9573071,2024-05-28 13:14:29,Orange Juice,Cold Drinks & Juices,Baral-Kamdar,Orange Juice | Baral-Kamdar,9573071_2024-05-28
2,9185164487,741341,2,923.84,45477575,2024-09-23 13:07:12,Eggs,Dairy & Breakfast,Prasad LLC,Eggs | Prasad LLC,45477575_2024-09-23
3,9644738826,561860,1,874.78,88067569,2023-11-24 16:16:56,Orange Juice,Cold Drinks & Juices,Gupta Ltd,Orange Juice | Gupta Ltd,88067569_2023-11-24
4,5427684290,602241,2,976.55,83298567,2023-11-20 05:00:39,Nuts,Snacks & Munchies,Bahl-Pau,Nuts | Bahl-Pau,83298567_2023-11-20
5,3265154092,681063,1,321.28,43367112,2023-03-18 16:29:51,Mango Drink,Cold Drinks & Juices,Dass and Sons,Mango Drink | Dass and Sons,43367112_2023-03-18
6,4898355547,56589,3,517.77,13284996,2023-04-16 18:50:37,Vitamins,Pharmacy,"Pai, Kashyap and Ramachandran","Vitamins | Pai, Kashyap and Ramachandran",13284996_2023-04-16
7,6568151549,500739,2,359.98,88866835,2024-03-31 06:26:48,Bread,Dairy & Breakfast,Aggarwal Group,Bread | Aggarwal Group,88866835_2024-03-31
8,6006693867,490939,1,80.25,24496983,2023-07-13 23:49:36,Lotion,Personal Care,"Roy, Barad and De","Lotion | Roy, Barad and De",24496983_2023-07-13
9,374186990,760828,2,563.45,52215833,2023-08-09 01:17:30,Pet Treats,Pet Care,"Khurana, Khosla and Yohannan","Pet Treats | Khurana, Khosla and Yohannan",52215833_2023-08-09


## Build Basket-Level Product Sets

In [13]:
basket_df = df_order_items_full_details.groupby('customer_id')['product_key'].apply(set)

# Map each product to all baskets it's in
product_to_baskets = defaultdict(set)
for order_id, items in basket_df.items():
    for item in items:
        product_to_baskets[item].add(order_id)

# Map each product to its complements (other products in same basket)
product_complements = defaultdict(set)
for items in basket_df:
    for item in items:
        others = items - {item}
        product_complements[item].update(others)

In [14]:
basket_df.head(15)

customer_id
31813     {Instant Noodles | Nadig, Zachariah and Soni, ...
61020     {Frozen Vegetables | Mammen-Hegde, Biscuits | ...
119099    {Detergent | Edwin-Lall, Cola | Bhandari, Bhas...
188838         {Eggs | Dasgupta PLC, Carrots | Uppal Group}
191616    {Onions | Aurora LLC, Biscuits | Chad, Yohanna...
211163    {Dish Soap | Arya, Sundaram and Pingle, Onions...
243838    {Potatoes | Chhabra-Agrawal, Cough Syrup | Ram...
376144    {Cheese | Kapadia-D’Alia, Sugar | Deo-Kamdar, ...
408590         {Toilet Cleaner | Gaba, Sodhi and Choudhary}
469006    {Vitamins | Kara-Golla, Dish Soap | Keer-Gill,...
625395    {Pulses | Balasubramanian PLC, Butter | Gara a...
644189                             {Shampoo | Malhotra LLC}
666589                   {Bananas | Doshi, Sarraf and Sama}
701493    {Wheat Flour | Gala, Magar and Rajagopal, Dog ...
767523           {Potatoes | Ramakrishnan, Anand and Khare}
Name: product_key, dtype: object

## Build Substitution Graph Network

Here we use a hybrid approach to identifying a valid substitute which comprises:
* A Shared Complements Score - If Product A and Product B are never bought together, but both are frequently bought with Product C, they likely serve the same function in the shopper's mind, thus making them substitutes
* A Mutual Exclusivity Score - where the edge weights reflect products rarely bought together but often bought individually

In [15]:
G = nx.Graph()
products = list(product_complements.keys())

for a, b in combinations(products, 2):
    # Skip if category mismatch
    if product_category.get(a) != product_category.get(b):
        continue

    # Co-purchases
    co_purchases = len(product_to_baskets[a] & product_to_baskets[b])
    count_a = len(product_to_baskets[a])
    count_b = len(product_to_baskets[b])

    # Mutual Exclusivity Score - here edge weights reflect products rarely bought together but often bought individually
    if min(count_a, count_b) == 0:
        continue
    mutual_exclusivity = 1 - (co_purchases / min(count_a, count_b))

    # Shared Complements Score - If Product A and Product B are never bought together, but both are frequently bought with Product C, 
    # ... they likely serve the same function in the shopper's mind, thus making them substitutes
    shared = product_complements[a] & product_complements[b]
    shared_complements_score = len(shared) / (1 + co_purchases)

    # Final hybrid score
    aa = 0.5
    bb = 0.5
    combined_score = aa * mutual_exclusivity + bb * shared_complements_score

    if combined_score >= 0.5:  # we can tune this threshold depending on our analysis of the two score weights across the dataset and time.
        G.add_edge(a, b, weight=combined_score, 
                   title=f"Excl: {round(mutual_exclusivity,2)}, Shared: {len(shared)}, Co: {co_purchases}")

Use a function to obtain the substiture for a given product brand.

In [ ]:
def recommend_substitutes_hybrid(
    product_key:str,
    G,
    product_meta:pd.DataFrame,
    top_n:int=5,
    match_category:bool=True,
    match_product_name:bool=False
):
    """
    Recommend top substitutes from the hybrid graph with optional filters.

    Args:
        product_key (str): e.g., "milk | Wali"
        G (networkx.Graph): Hybrid substitution graph
        product_meta (pd.DataFrame): Metadata with product_name, brand, category (indexed by product_key)
        top_n (int): Number of substitutes to return
        match_category (bool): Whether to restrict to same category
        match_product_name (bool): Whether to restrict to same product_name (different brand)

    Returns:
        List of tuples: (substitute, score, tooltip)
    """
    if product_key not in G:
        print("Product not found in graph.")
        return []

    try:
        target_meta = product_meta.loc[product_key]
    except KeyError:
        print("Product metadata not found.")
        return []

    target_category = target_meta['category']
    target_name = target_meta['product_name']
    target_brand = target_meta['brand']

    candidates = []
    for neighbor in G[product_key]:
        try:
            neighbor_meta = product_meta.loc[neighbor]
        except KeyError:
            continue

        if match_category and neighbor_meta['category'] != target_category:
            continue
        if match_product_name and neighbor_meta['product_name'] != target_name:
            continue
        if match_product_name and neighbor_meta['brand'] == target_brand:
            continue  # avoids same-brand if doing brand-level substitution

        score = G[product_key][neighbor]['weight']
        tooltip = G[product_key][neighbor].get('title', '')
        candidates.append((neighbor, score, tooltip))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:top_n]

In [18]:
product = "Orange Juice | Gupta Ltd" #"Eggs | Prasad LLC" # "Bread | Aggarwal Group" # "Orange Juice | Baral-Kamdar" # "Milk | Wali, Virk and Iyer" # 
swap_recommendations = recommend_substitutes_hybrid(product, G, product_meta, match_category=True, match_product_name=True)

# "Milk | Wali, Virk and Iyer" is the only brand of milk, so there woudl be no subsitutes.

In [19]:
swap_recommendations

[('Orange Juice | D’Alia-Dey', 4.0, 'Excl: 1.0, Shared: 7, Co: 0'),
 ('Orange Juice | Chana LLC', 4.0, 'Excl: 1.0, Shared: 7, Co: 0'),
 ('Orange Juice | Palla LLC', 2.5, 'Excl: 1.0, Shared: 4, Co: 0'),
 ('Orange Juice | Toor-Nagar', 2.5, 'Excl: 1.0, Shared: 4, Co: 0'),
 ('Orange Juice | Baral-Kamdar', 2.5, 'Excl: 1.0, Shared: 4, Co: 0')]

## Comments

There are numerous other methods we could use, e.g. collaborative filtering baased on a uni-partite projection (product space) of a bi-partite network (customer partition and product partition), such that our recommendations of products are not based solely on the product purchasing behaviour alone, but also factor in common-products across customers to build customer cohort profiles and recommend economic substitutes or compliments thus allowing us to upsell or swap in a manner that would be satisfactory to the cutomers.